# Quasi-Harmonic Free Energy Update

Objective: recompute thermochemical free energies from the Gaussian optimization/frequency logs with AaronTools low-frequency treatments, then add the converted free-energy entries back into the ASE database.

This notebook follows the raw-log mapping in `3_Build_DataBase.ipynb`. It does not apply the standard-state correction; that is handled in the SI statement.

## Plan

- Check that `main_py3_12` can import AaronTools and that `CompOutput` can compute quasi-harmonic corrections.
- Build the same key-to-log-file map used by `3_Build_DataBase.ipynb`.
- Run one small timing test before the full pass.
- Compute RRHO and quasi-harmonic free energies for every available optimization/frequency log.
- Add derived fields to a copy of `BorylXAT-DB.db`, update the JACS experimental database in place, and export an audit CSV.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import json
import os
import platform
import re
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from DFTStructureGenerator.project_paths import CSV_DIR, TS_DATA_DIR, RAW_CALC_ROOT, raw_calc_file

HARTREE_TO_KCAL = 627.509474
TEMPERATURE_K = 298.15
FREQUENCY_CUTOFF_CM = 100.0
SOURCE_DB = Path("BorylXAT-DB.db")
OUTPUT_DB = Path("BorylXAT-DB_qh_update.db")
JACS_EXPERIMENT_DB = Path("output/jacs_experiment/BorylXAT-DB_with_jacs_experiment.db")
OUTPUT_CSV = Path("output/jupyter-notebook/quasi_harmonic_free_energy_update.csv")
OUTPUT_SUMMARY_CSV = Path("output/jupyter-notebook/quasi_harmonic_reaction_energy_summary.csv")

pd.set_option("display.max_columns", 120)
print("Python:", platform.python_version())
print("Raw Gaussian root:", RAW_CALC_ROOT)

Python: 3.12.7
Raw Gaussian root: E:\work\B_Cl_Nu


c:\Users\Jackie\anaconda3\envs\main_py3_12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Local Key Parser

This mirrors `DFTStructureGenerator.Build_DataBase.classify_key` without importing ASE during setup. ASE is only needed in the final database update cell.

In [2]:
PATTERNS = {
    "B": re.compile(r"^B_(\d{5})$"),
    "LB": re.compile(r"^LB_(\d{5})$"),
    "Cl": re.compile(r"^Cl_(\d{5})_r$"),
    "complex_r": re.compile(r"^B_(\d{5})_LB_(\d{5})_r$"),
    "complex_p": re.compile(r"^B_(\d{5})_LB_(\d{5})_p$"),
    "ts": re.compile(r"^B_(\d{5})_LB_(\d{5})_Cl_(\d{5})$"),
    "c_radical": re.compile(r"^Cl_(\d{5})_p$"),
}


def classify_key(key):
    for pat, regex in PATTERNS.items():
        match = regex.match(key)
        if not match:
            continue
        groups = [int(g) for g in match.groups()]
        if pat == "B":
            return "B", (groups[0], np.nan, np.nan)
        if pat == "LB":
            return "LB", (np.nan, groups[0], np.nan)
        if pat == "Cl":
            return "Cl", (np.nan, np.nan, groups[0])
        if pat == "complex_r":
            return "complex_r", (groups[0], groups[1], np.nan)
        if pat == "complex_p":
            return "complex_p", (groups[0], groups[1], np.nan)
        if pat == "ts":
            return "ts", (groups[0], groups[1], groups[2])
        if pat == "c_radical":
            return "c_radical", (np.nan, np.nan, groups[0])
    return "unknown", None

## AaronTools Preflight

AaronTools has the needed API: `CompOutput.calc_G_corr(method="QHARM")` for Truhlar-style quasi-harmonic entropy and `CompOutput.calc_G_corr(method="QRRHO")`/`calc_Grimme_G` for Grimme quasi-RRHO. This cell verifies whether the current kernel environment can actually import it.

In [3]:
def check_aarontools():
    info = {"ok": False}
    try:
        import AaronTools
        info["aarontools_file"] = getattr(AaronTools, "__file__", None)
        info["aarontools_version"] = getattr(AaronTools, "__version__", None)
        from AaronTools.comp_output import CompOutput
        info["CompOutput"] = CompOutput
        info["ok"] = True
    except Exception as exc:
        info["error_type"] = type(exc).__name__
        info["error"] = str(exc)
        info["traceback"] = traceback.format_exc(limit=6)
    return info


aarontools_status = check_aarontools()
print(json.dumps({k: v for k, v in aarontools_status.items() if k != "CompOutput"}, indent=2, default=str))

if not aarontools_status["ok"]:
    print("\nAaronTools cannot be used in this kernel yet. Fix the environment first, then rerun from this cell.")
    print("Observed in current test environment: SciPy may have mixed package files if scipy.__version__ and pip metadata disagree.")

{
  "ok": true,
  "aarontools_file": "c:\\Users\\Jackie\\anaconda3\\envs\\main_py3_12\\Lib\\site-packages\\AaronTools\\__init__.py",
  "aarontools_version": null
}


## Build Log File Map

This reproduces the log-file choices in `3_Build_DataBase.ipynb`. Each key matches the ASE database key, so the update step can join directly by `key`.

In [5]:
duplicate_N_id = [9, 43, 285, 310, 314, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 372, 375, 376]

Borane_df = pd.read_csv(CSV_DIR / "reactants_B.csv")
nu_df = pd.read_csv(CSV_DIR / "reactants_N.csv").dropna(subset=["G_energy_r"])
Cl_df = pd.read_csv(CSV_DIR / "reactants_Cl.csv")
Borane_nu_df = pd.read_csv(CSV_DIR / "reactants_B_N.csv")
reaction_df = pd.read_csv(TS_DATA_DIR / "Borane_all.csv")


def build_log_manifest():
    rows = []

    for _, row in Borane_df.iterrows():
        idx = int(row["Index"])
        conf = int(row["conf_idxs_r"])
        rows.append({
            "key": f"B_{idx:05}",
            "category": "B",
            "source_gibbs_hartree": row["G_energy_r"],
            "log_file": raw_calc_file("Data", "GS_OPT", "B_single", f"B_{idx:05}_r_{conf:04}.log"),
        })

    for _, row in nu_df.iterrows():
        idx = int(row["Index"])
        conf = int(row["conf_idxs_r"])
        rows.append({
            "key": f"LB_{idx:05}",
            "category": "LB",
            "source_gibbs_hartree": row["G_energy_r"],
            "log_file": raw_calc_file("Data", "GS_OPT", "N_single", f"N_{idx:05}_r_{conf:04}.log"),
        })

    for _, row in Cl_df.iterrows():
        idx = int(row["Index"])
        for conf, state, G in [
            (int(row["conf_idxs_r"]), "r", row["G_energy_r"]),
            (int(row["conf_idxs_p"]), "p", row["G_energy_p"]),
        ]:
            rows.append({
                "key": f"Cl_{idx:05}_{state}",
                "category": "Cl" if state == "r" else "c_radical",
                "source_gibbs_hartree": G,
                "log_file": raw_calc_file("Data", "GS_OPT", f"Cl_{state}", f"Cl_{idx:05}_{state}_{conf:04}.log"),
            })

    for _, row in reaction_df.iterrows():
        B_idx = int(row["B_Index"])
        N_idx = int(row["N_Index"])
        Cl_idx = int(row["Cl_Index"])
        conf = int(row["conf_idxs_ts"])
        base = raw_calc_file("Sum", "TS_needIRC", f"B_{B_idx:05}_Nu_{N_idx:05}_Cl_{Cl_idx:05}.log")
        fallback = raw_calc_file("Sum", "TS_needIRC", f"B_{B_idx:05}_Nu_{N_idx:05}_Cl_{Cl_idx:05}_{conf:04}.log")
        rows.append({
            "key": f"B_{B_idx:05}_LB_{N_idx:05}_Cl_{Cl_idx:05}",
            "category": "ts",
            "source_gibbs_hartree": row["TS_G"],
            "log_file": base if Path(base).exists() else fallback,
        })

    for _, row in Borane_nu_df.iterrows():
        B_idx = int(row["B_Index"])
        N_idx = int(row["N_Index"])
        N_atom = int(row["N_Atomid"])
        for conf, state, G in [
            (int(row["conf_idxs_r"]), "r", row["G_energy_r"]),
            (int(row["conf_idxs_p"]), "p", row["G_energy_p"]),
        ]:
            folder = f"B_N_{state}"
            filename = f"B_{B_idx:05}_Nu_{N_idx:05}_{state}_{conf:04}.log"
            if N_idx in duplicate_N_id:
                folder = f"B_N_{state}_d"
                filename = f"B_{B_idx:05}_Nu_{N_idx:05}_Naid_{N_atom:05}_{state}_{conf:04}.log"
            rows.append({
                "key": f"B_{B_idx:05}_LB_{N_idx:05}_{state}",
                "category": f"complex_{state}",
                "source_gibbs_hartree": G,
                "log_file": raw_calc_file("Data", "GS_OPT", folder, filename),
            })

    manifest = pd.DataFrame(rows)
    manifest["log_file"] = manifest["log_file"].map(str)
    manifest["log_exists"] = manifest["log_file"].map(lambda p: Path(p).exists())
    return manifest


log_manifest = build_log_manifest()
display(log_manifest.groupby(["category", "log_exists"]).size().unstack(fill_value=0))
display(log_manifest.head())

log_exists,True
category,
B,55
Cl,181
LB,413
c_radical,181
complex_p,20010
complex_r,20010
ts,8988


,key,category,source_gibbs_hartree,log_file,log_exists
0,B_00388,B,-25.931835,E:\work\B_Cl_Nu\Data\GS_OPT\B_single\B_00388_r...,True
1,B_00389,B,-118.189505,E:\work\B_Cl_Nu\Data\GS_OPT\B_single\B_00389_r...,True
2,B_00390,B,-125.260723,E:\work\B_Cl_Nu\Data\GS_OPT\B_single\B_00390_r...,True
3,B_00391,B,-102.092050,E:\work\B_Cl_Nu\Data\GS_OPT\B_single\B_00391_r...,True
4,B_00392,B,-65.243012,E:\work\B_Cl_Nu\Data\GS_OPT\B_single\B_00392_r...,True


## Conversion Function

Definitions used below:

- `gibbs_rrho_hartree`: AaronTools electronic energy plus RRHO free-energy correction.
- `gibbs_qharm_hartree`: electronic energy plus quasi-harmonic free-energy correction, where positive frequencies below `FREQUENCY_CUTOFF_CM` are raised to the cutoff in the entropy term.
- `gibbs_qrrho_hartree`: electronic energy plus Grimme quasi-RRHO correction.
- `gibbs_full_qrrho_hartree`: AaronTools' full quasi-RRHO variant, using quasi-RRHO treatment in both entropy and enthalpy.

In [10]:
def compute_aarontools_thermo(log_file, temperature=TEMPERATURE_K, cutoff_cm=FREQUENCY_CUTOFF_CM):
    if not aarontools_status["ok"]:
        raise RuntimeError("AaronTools CompOutput is not importable in this kernel. Rerun the preflight after fixing the environment.")

    CompOutput = aarontools_status["CompOutput"]
    co = CompOutput(str(log_file))
    if co.frequency is None:
        raise ValueError("No frequency block found")
    if co.energy is None:
        raise ValueError("No electronic energy found")

    rrho_dG = co.calc_G_corr(temperature=temperature, v0=0, method="RRHO")
    qharm_dG = co.calc_G_corr(temperature=temperature, v0=cutoff_cm, method="QHARM")
    qrrho_dG = co.calc_G_corr(temperature=temperature, v0=cutoff_cm, method="QRRHO")
    _, qrrho_dH, qrrho_entropy = co.therm_corr(
        temperature=temperature,
        v0=cutoff_cm,
        method="QRRHO",
        enthalpy_method="QRRHO",
    )
    full_qrrho_dG = qrrho_dH - temperature * qrrho_entropy

    freqs = np.array(co.frequency.real_frequencies, dtype=float)
    positive_freqs = freqs[freqs > 0]
    low_positive_freqs = positive_freqs[positive_freqs < cutoff_cm]

    return {
        "electronic_energy_hartree": float(co.energy),
        "gibbs_rrho_hartree": float(co.energy + rrho_dG),
        "gibbs_qharm_hartree": float(co.energy + qharm_dG),
        "gibbs_qrrho_hartree": float(co.energy + qrrho_dG),
        "gibbs_full_qrrho_hartree": float(co.energy + full_qrrho_dG),
        "rrho_g_corr_hartree": float(rrho_dG),
        "qharm_g_corr_hartree": float(qharm_dG),
        "qrrho_g_corr_hartree": float(qrrho_dG),
        "full_qrrho_g_corr_hartree": float(full_qrrho_dG),
        "temperature_K": float(temperature),
        "frequency_cutoff_cm_1": float(cutoff_cm),
        "n_real_frequencies": int(len(freqs)),
        "n_low_positive_frequencies": int(len(low_positive_freqs)),
        "min_positive_frequency_cm_1": float(np.min(positive_freqs)) if len(positive_freqs) else np.nan,
        "low_positive_frequencies_cm_1": low_positive_freqs.tolist(),
    }

## One-Log Test and Timing

Run this before the full pass. It validates the parser and gives a rough per-log timing estimate.

In [11]:
test_candidates = log_manifest[log_manifest["log_exists"]].copy()
test_candidates["priority"] = np.where(test_candidates["category"].eq("ts"), 0, 1)
test_row = test_candidates.sort_values(["priority", "key"]).head(1).squeeze()
print(test_row[["key", "category", "log_file"]].to_string())

if aarontools_status["ok"] and len(test_candidates):
    t0 = time.perf_counter()
    test_result = compute_aarontools_thermo(test_row["log_file"])
    elapsed = time.perf_counter() - t0
    print(f"Elapsed: {elapsed:.3f} s for one log")
    display(pd.Series(test_result).drop(labels=["low_positive_frequencies_cm_1"]).to_frame("value"))
else:
    print("Skipped: AaronTools preflight failed or no raw logs were found.")

key                                 B_00388_LB_00000_Cl_00500
category                                                   ts
log_file    E:\work\B_Cl_Nu\Sum\TS_needIRC\B_00388_Nu_0000...
Elapsed: 0.224 s for one log


,value
electronic_energy_hartree,-1030.001166
gibbs_rrho_hartree,-1029.671208
gibbs_qharm_hartree,-1029.66598
gibbs_qrrho_hartree,-1029.666892
gibbs_full_qrrho_hartree,-1029.670358
rrho_g_corr_hartree,0.329958
qharm_g_corr_hartree,0.335186
qrrho_g_corr_hartree,0.334275
full_qrrho_g_corr_hartree,0.330808
temperature_K,298.15


## Full Conversion

In [12]:
def convert_manifest(manifest, limit=None):
    rows = []
    work = manifest[manifest["log_exists"]].copy()
    if limit is not None:
        work = work.head(limit)

    for rec in tqdm(work.to_dict("records"), total=len(work)):
        row = {**rec, "status": "ok", "error": None}
        try:
            row.update(compute_aarontools_thermo(rec["log_file"]))
        except Exception as exc:
            row["status"] = "failed"
            row["error"] = f"{type(exc).__name__}: {exc}"
        rows.append(row)

    missing = manifest[~manifest["log_exists"]].copy()
    if len(missing):
        missing["status"] = "missing_log"
        missing["error"] = "log_file does not exist"
        rows.extend(missing.to_dict("records"))

    return pd.DataFrame(rows)


RUN_FULL_CONVERSION = True

if RUN_FULL_CONVERSION:
    t0 = time.perf_counter()
    thermo_df = convert_manifest(log_manifest)
    print(f"Converted {len(thermo_df)} rows in {(time.perf_counter() - t0) / 60:.2f} min")
else:
    print("RUN_FULL_CONVERSION is False. Set it to True after the one-log test passes.")
    thermo_df = pd.DataFrame()

 20%|██        | 10085/49838 [21:43<1:52:03,  5.91it/s]WARNING AaronTools.fileIO.FileReader.read_log 
  error parsing forces:
 could not convert string to float: 
  '7267.65506240124464.37198242129925.433393051'
 30%|███       | 15180/49838 [31:59<2:21:19,  4.09it/s]WARNING AaronTools.fileIO.FileReader.read_log 
  error parsing forces:
 could not convert string to float: 
  '*********************************************'
 36%|███▋      | 18102/49838 [38:28<1:15:33,  7.00it/s]WARNING AaronTools.fileIO.FileReader.read_log 
  error parsing forces:
 could not convert string to float: 
  '*********************************************'
 40%|████      | 20038/49838 [42:45<1:44:43,  4.74it/s] WARNING AaronTools.fileIO.FileReader.read_log 
  error parsing forces:
 could not convert string to float: 
  '16023.041387506-9670.084706430***************'
 42%|████▏     | 20902/49838 [44:37<1:50:07,  4.38it/s]WARNING AaronTools.fileIO.FileReader.read_log 
  error parsing forces:
 could not convert str

Converted 49838 rows in 115.93 min


## Save Audit Table

In [13]:
if not thermo_df.empty:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    thermo_df.to_csv(OUTPUT_CSV, index=False)
    display(thermo_df.groupby(["category", "status"]).size().unstack(fill_value=0))
    print("Wrote", OUTPUT_CSV)
else:
    print("No conversion table in memory yet.")

status,ok
category,
B,55
Cl,181
LB,413
c_radical,181
complex_p,20010
complex_r,20010
ts,8988


Wrote output\jupyter-notebook\quasi_harmonic_free_energy_update.csv


## Reaction-Level Energies from Converted Free Energies

This mirrors the second pass in `build_databases`, but uses `gibbs_qharm_hartree` and keeps the original database untouched until the explicit update cell.

In [14]:
thermo_df = pd.read_csv(OUTPUT_CSV)

In [15]:
def build_reaction_energy_summary(thermo_df, gibbs_col="gibbs_qharm_hartree"):
    ok = thermo_df[thermo_df["status"].eq("ok")].copy()
    g = ok.set_index("key")[gibbs_col].to_dict()
    rows = []
    for key in sorted(ok.loc[ok["category"].eq("ts"), "key"]):
        category, ids = classify_key(key)
        if category != "ts":
            continue
        bid, lid, clid = [int(x) for x in ids]
        b5, l5, c5 = f"{bid:05}", f"{lid:05}", f"{clid:05}"
        r_complex = f"B_{b5}_LB_{l5}_r"
        r_cl = f"Cl_{c5}_r"
        p_complex = f"B_{b5}_LB_{l5}_p"
        p_rad = f"Cl_{c5}_p"
        barrier = np.nan
        delta_g = np.nan
        if all(k in g for k in [key, r_complex, r_cl]):
            barrier = (g[key] - g[r_complex] - g[r_cl]) * HARTREE_TO_KCAL
        if all(k in g for k in [p_complex, p_rad, r_complex, r_cl]):
            delta_g = (g[p_complex] + g[p_rad] - g[r_complex] - g[r_cl]) * HARTREE_TO_KCAL
        rows.append({
            "key": key,
            "barrier_qharm_kcal": barrier,
            "delta_g_rxn_qharm_kcal": delta_g,
            "reactant_complex_key": r_complex,
            "reactant_cl_key": r_cl,
            "product_complex_key": p_complex,
            "product_c_radical_key": p_rad,
        })
    return pd.DataFrame(rows)


if not thermo_df.empty:
    reaction_qharm_df = build_reaction_energy_summary(thermo_df, gibbs_col="gibbs_qharm_hartree")
    reaction_qharm_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
    display(reaction_qharm_df[["barrier_qharm_kcal", "delta_g_rxn_qharm_kcal"]].describe())
    print("Wrote", OUTPUT_SUMMARY_CSV)
else:
    reaction_qharm_df = pd.DataFrame()
    print("No conversion table in memory yet.")

,barrier_qharm_kcal,delta_g_rxn_qharm_kcal
count,8988.000000,8988.000000
mean,15.247055,-20.217217
std,5.051757,9.943032
min,-3.385036,-64.300651
25%,11.641805,-27.094788
50%,15.221386,-19.271063
75%,18.726831,-12.157033
max,40.354847,-4.137677


Wrote output\jupyter-notebook\quasi_harmonic_reaction_energy_summary.csv


## Update the QHARM and JACS Experimental Databases

This cell adds the QHARM key-value entries to both database targets in one ASE transaction per database. ASE lock files are disabled because these databases live in a OneDrive-synced directory, where Windows may hold `.db.lock` long enough to cause `WinError 32`; SQLite's transaction lock still protects this single-process write. The complete `BorylXAT-DB_qh_update.db` is used as the authoritative QHARM source for all keys shared with the JACS database, while `thermo_df` supplies JACS-only rows such as Cl 723/724. This preserves the experimental rows without dropping older QHARM records that are absent from a newly rebuilt manifest.

In [19]:
def update_database(thermo_df, reaction_df, source_db, output_db, target_name, skip_if_complete=False, seed_qharm_db=None):
    import sqlite3
    from ase.db import connect

    source_db = Path(source_db)
    output_db = Path(output_db)
    if not source_db.exists():
        raise FileNotFoundError(source_db)
    if output_db.resolve() != source_db.resolve() and not output_db.exists():
        output_db.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_db, output_db)

    db = connect(output_db, use_lock_file=False)
    total_rows = db.count()
    qharm_rows = db.count("gibbs_qharm_hartree")
    if skip_if_complete and qharm_rows == total_rows:
        return {
            "target": target_name,
            "output_db": str(output_db),
            "skipped": "all database rows already have gibbs_qharm_hartree",
            "qharm_rows": qharm_rows,
            "total_rows": total_rows,
        }

    ok = thermo_df[thermo_df["status"].eq("ok")].copy()
    updated_structure_keys = set()
    seeded_reaction_keys = set()
    thermo_keys_absent_from_db = set()
    updated_ts = 0
    if seed_qharm_db is not None:
        seed_qharm_db = Path(seed_qharm_db)
        if not seed_qharm_db.exists():
            raise FileNotFoundError(seed_qharm_db)
        qharm_fields = [
            "gibbs_rrho_aarontools_hartree",
            "gibbs_qharm_hartree",
            "gibbs_qrrho_hartree",
            "gibbs_full_qrrho_hartree",
            "qharm_delta_vs_original_kcal",
            "qharm_frequency_cutoff_cm_1",
            "qharm_temperature_K",
            "qharm_n_low_positive_frequencies",
            "barrier_qharm_kcal",
            "delta_g_rxn_qharm_kcal",
        ]
        sql_db = sqlite3.connect(output_db, timeout=60)
        sql_db.execute("PRAGMA busy_timeout=60000")
        sql_db.execute("ATTACH DATABASE ? AS qh_seed", (str(seed_qharm_db.resolve()),))
        seed_id_to_key = dict(sql_db.execute("SELECT id, value FROM qh_seed.text_key_values WHERE key='key'"))
        target_id_to_key = dict(sql_db.execute("SELECT id, value FROM main.text_key_values WHERE key='key'"))
        mismatched_ids = [row_id for row_id, key in seed_id_to_key.items() if target_id_to_key.get(row_id) != key]
        if mismatched_ids:
            sql_db.close()
            raise RuntimeError(f"QHARM/JACS row-ID mismatch; first IDs: {mismatched_ids[:10]}")
        seeded_reaction_ids = {
            row_id for (row_id,) in sql_db.execute("SELECT id FROM qh_seed.number_key_values WHERE key='barrier_qharm_kcal'")
        }
        placeholders = ",".join("?" for _ in qharm_fields)
        try:
            sql_db.execute("BEGIN IMMEDIATE")
            sql_db.execute(
                "UPDATE main.systems SET key_value_pairs=("
                "SELECT src.key_value_pairs FROM qh_seed.systems src WHERE src.id=main.systems.id"
                ") WHERE id IN (SELECT id FROM qh_seed.systems)"
            )
            sql_db.execute(f"DELETE FROM main.number_key_values WHERE key IN ({placeholders})", qharm_fields)
            sql_db.execute(
                f"INSERT INTO main.number_key_values(key, value, id) "
                f"SELECT key, value, id FROM qh_seed.number_key_values WHERE key IN ({placeholders})",
                qharm_fields,
            )
            sql_db.execute(f"DELETE FROM main.keys WHERE key IN ({placeholders})", qharm_fields)
            sql_db.execute(
                f"INSERT INTO main.keys(key, id) "
                f"SELECT key, id FROM qh_seed.keys WHERE key IN ({placeholders})",
                qharm_fields,
            )
            sql_db.commit()
        except Exception:
            sql_db.rollback()
            raise
        finally:
            sql_db.close()
        updated_structure_keys.update(seed_id_to_key.values())
        seeded_reaction_keys.update(seed_id_to_key[row_id] for row_id in seeded_reaction_ids)
        updated_ts = len(seeded_reaction_keys)

    with db:
        for rec in tqdm(ok.to_dict("records"), total=len(ok), desc=f"{target_name}: structures"):
            if rec["key"] in updated_structure_keys:
                continue
            try:
                row = db.get(key=rec["key"])
            except KeyError:
                thermo_keys_absent_from_db.add(rec["key"])
                continue
            db.update(
                row.id,
                gibbs_rrho_aarontools_hartree=float(rec["gibbs_rrho_hartree"]),
                gibbs_qharm_hartree=float(rec["gibbs_qharm_hartree"]),
                gibbs_qrrho_hartree=float(rec["gibbs_qrrho_hartree"]),
                gibbs_full_qrrho_hartree=float(rec["gibbs_full_qrrho_hartree"]),
                qharm_delta_vs_original_kcal=float((rec["gibbs_qharm_hartree"] - rec["source_gibbs_hartree"]) * HARTREE_TO_KCAL),
                qharm_frequency_cutoff_cm_1=float(rec["frequency_cutoff_cm_1"]),
                qharm_temperature_K=float(rec["temperature_K"]),
                qharm_n_low_positive_frequencies=int(rec["n_low_positive_frequencies"]),
            )
            updated_structure_keys.add(rec["key"])

        if not reaction_df.empty:
            for rec in tqdm(reaction_df.to_dict("records"), total=len(reaction_df), desc=f"{target_name}: TS rows"):
                if rec["key"] in seeded_reaction_keys:
                    continue
                try:
                    row = db.get(key=rec["key"])
                except KeyError:
                    continue
                db.update(
                    row.id,
                    barrier_qharm_kcal=float(rec["barrier_qharm_kcal"]) if pd.notna(rec["barrier_qharm_kcal"]) else np.nan,
                    delta_g_rxn_qharm_kcal=float(rec["delta_g_rxn_qharm_kcal"]) if pd.notna(rec["delta_g_rxn_qharm_kcal"]) else np.nan,
                )
                updated_ts += 1

    missing_qharm_keys = [
        row.key for row in db.select()
        if "gibbs_qharm_hartree" not in row.key_value_pairs
    ]

    return {
        "target": target_name,
        "output_db": str(output_db),
        "updated_structure_keys": len(updated_structure_keys),
        "thermo_keys_absent_from_db": sorted(thermo_keys_absent_from_db),
        "updated_ts_reaction_rows": updated_ts,
        "database_keys_without_qharm": sorted(missing_qharm_keys),
        "total_rows": db.count(),
    }


WRITE_DB_UPDATE = True
DATABASE_UPDATE_TARGETS = [
    {"target_name": "qharm_copy", "source_db": SOURCE_DB, "output_db": OUTPUT_DB, "skip_if_complete": True},
    {"target_name": "jacs_experiment_in_place", "source_db": JACS_EXPERIMENT_DB, "output_db": JACS_EXPERIMENT_DB, "seed_qharm_db": OUTPUT_DB, "skip_if_complete": True},
]

if WRITE_DB_UPDATE and not thermo_df.empty:
    update_summaries = [
        update_database(thermo_df, reaction_qharm_df, **target)
        for target in DATABASE_UPDATE_TARGETS
    ]
    print(json.dumps(update_summaries, indent=2))
else:
    print("WRITE_DB_UPDATE is False or no conversion table is available.")

[
  {
    "target": "qharm_copy",
    "output_db": "BorylXAT-DB_qh_update.db",
    "skipped": "all database rows already have gibbs_qharm_hartree",
    "qharm_rows": 50057,
    "total_rows": 50057
  },
  {
    "target": "jacs_experiment_in_place",
    "output_db": "output\\jacs_experiment\\BorylXAT-DB_with_jacs_experiment.db",
    "skipped": "all database rows already have gibbs_qharm_hartree",
    "qharm_rows": 50061,
    "total_rows": 50061
  }
]


## Reload Existing Audit CSV

Use this section when the full conversion has already been run and only the database update or summaries need to be rerun.

In [8]:
if OUTPUT_CSV.exists():
    saved_thermo_df = pd.read_csv(OUTPUT_CSV)
    display(saved_thermo_df.groupby(["category", "status"]).size().unstack(fill_value=0))
else:
    saved_thermo_df = pd.DataFrame()
    print("No saved audit CSV found yet:", OUTPUT_CSV)

No saved audit CSV found yet: output\jupyter-notebook\quasi_harmonic_free_energy_update.csv


## Notes for Manuscript/SI

- The notebook computes low-frequency corrected free energies from the same optimized Gaussian log files used to build the released database.
- The default cutoff is 100 cm^-1 at 298.15 K, matching AaronTools defaults for quasi treatments.
- Use `gibbs_qharm_hartree`, `barrier_qharm_kcal`, and `delta_g_rxn_qharm_kcal` for the quasi-harmonic sensitivity analysis.
- The standard-state correction is intentionally not applied here.